# Thực nghiệm crawl TGDD

Mục đích: thử selector, hiểu parser và xem dữ liệu đã lấy. Notebook không truy cập mạng.


In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd

root = Path.cwd()
if not (root / 'main.py').exists():
    root = root.parent
if not (root / 'main.py').exists():
    raise RuntimeError('Mở notebook từ thư mục tgdd_crawler hoặc notebooks.')
sys.path.insert(0, str(root))

from config import DATA_DIR, PRODUCT_SELECTOR, NAME_SELECTOR, PRICE_SELECTOR
from parser import parse_products, convert_price
print('Python:', sys.executable)
print('Project:', root)

## 2. Thử cách đổi giá

Giá bán rõ ràng chuyển thành số. Giá thiếu hoặc trả góp để None, không thay bằng 0.

In [ ]:
prices = ['12.990.000₫', '', 'Liên hệ', 'Từ 1.000.000₫/tháng']
for price in prices:
    print(repr(price), '->', convert_price(price))

## 3. Thử parser bằng HTML mẫu

Đây là mẫu tự tạo, không phải dữ liệu TGDD. Không lưu mẫu này thành dữ liệu nộp bài.

In [ ]:
sample_html = """
<ul class="listproduct">
  <li class="item"><a href="/dtdd/may-mau-a">
    <h3>Máy mẫu A</h3><strong class="price">12.990.000₫</strong>
  </a></li>
  <li class="item"><a href="/dtdd/may-mau-b"><h3>Máy mẫu B</h3></a></li>
</ul>
"""
sample_products = parse_products(sample_html)
display(pd.DataFrame(sample_products))
assert sample_products[0]['price_vnd'] == 12990000
assert sample_products[1]['price_vnd'] is None

## 4. Đọc HTML đã crawl

Chạy main.py crawl ở terminal trước. Nếu chưa có page.html, ô này chỉ in hướng dẫn.

Chọn một khối sản phẩm để nhìn tên/giá và sửa selector trong config.py nếu cần. Sau khi sửa file Python, restart kernel rồi chạy lại các ô.

In [ ]:
from bs4 import BeautifulSoup

html_file = DATA_DIR / 'page.html'
html = None
if html_file.exists():
    html = html_file.read_text(encoding='utf-8')
    soup = BeautifulSoup(html, 'html.parser')
    cards = soup.select(PRODUCT_SELECTOR)
    print('Số khối sản phẩm:', len(cards))
    if cards:
        print(str(cards[0])[:2000])
else:
    print('Chưa có HTML. Chạy main.py crawl trong terminal trước.')

## 5. Parse và kiểm tra dữ liệu thật

Đối chiếu 5 sản phẩm trên website. Giá thiếu có thể do sản phẩm không công bố giá hoặc selector chưa đúng. Số bản ghi không tự chứng minh đã lấy hết danh mục.

In [ ]:
if html is not None:
    products = parse_products(html)
    df = pd.DataFrame(products)
    display(df.head())
    print('Số sản phẩm:', len(df))
    print('Thiếu giá:', df['price_vnd'].isna().sum())
    print('URL trùng:', df['url'].duplicated().sum())
else:
    print('Chưa có HTML thật để parse.')

## 6. Xem file kết quả

File JSON được main.py tạo từ dữ liệu crawl. Notebook chỉ đọc, không ghi đè file.

In [ ]:
json_file = DATA_DIR / 'products.json'
if json_file.exists():
    saved_products = json.loads(json_file.read_text(encoding='utf-8'))
    display(pd.DataFrame(saved_products).head(10))
else:
    print('Chưa có products.json. Chạy main.py crawl hoặc main.py parse.')

## 7. Tự ghi nhận kết quả

- Selector nào đã xác minh đúng?
- Thu thập lúc nào và chọn khu vực nào?
- Đã bấm Xem thêm bao nhiêu lần?
- Có bao nhiêu sản phẩm, bao nhiêu sản phẩm thiếu giá?
- Có lấy nhầm giá cũ hoặc giá trả góp không?
- Chương trình dừng vì đủ số lượt, hết nút hay lỗi?
- Cần sửa gì để phù hợp cấu trúc website thực tế?